In [ ]:
# ── 第一格：环境准备 ──
%load_ext autoreload
%autoreload 2
from signal_mysql import database as mysql
import pandas as pd
from signal_insight import config
DAY = "2018-10-03"

In [ ]:
# 信息架构：谁看、看什么。7 页 = 总览 + 6 个分页；每页只回答一个问题。
pages = {"总览": "今天整体怎么样：傍晚访客、高峰、客源距离、删失",
         "客流节律": "每个区域几点人多",
         "客源与圈层": "客群构成、住多远、从哪个区县来",
         "商圈联动": "先去哪儿、后去哪儿",
         "空间分布": "人在城市的哪些格子里",
         "数据质量": "哪些分区被门禁拦下、为什么",
         "指标口径": "每个数字是怎么算的"}
pd.DataFrame(pages.items(), columns=["页面", "回答的问题"])

In [ ]:
# 数据访问层：平台只读 ads_ 和 dim_。这里直接在 notebook 里调用它。
import sys
from pathlib import Path
sys.path.insert(0, str(config.PROJECT_ROOT / "app"))
from lib import queries as Q
con = Q.connect()
print("可用日期：", Q.dates(con))
print("可用区域：", Q.areas(con))
Q.area_daily(con, DAY)

In [ ]:
Q.area_hourly(con, DAY).head(24)

In [ ]:
Q.area_segment(con, DAY)

In [ ]:
Q.area_radius(con, DAY)

In [ ]:
Q.area_linkage(con, DAY)

In [ ]:
Q.dq_partition(con)

In [ ]:
# 冒烟测试：用假的 streamlit / plotly 把 7 页从头执行到尾，不需要浏览器。
import subprocess, sys
r = subprocess.run([sys.executable, "-m", "pytest", "-q", "tests/integration/test_app_smoke.py"],
                   cwd=config.PROJECT_ROOT, capture_output=True, text=True, encoding="utf-8")
print(r.stdout[-800:])

In [ ]:
# 部署：对外版生产库只导出 ads_ / dim_；Docker 只放代码和聚合结果。
print((config.PROJECT_ROOT / "Dockerfile").read_text(encoding="utf-8"))
con.close()